## 1. Configuration — update these paths

In [ ]:
from pathlib import Path
import os, sys, subprocess, csv
import torch

# UPDATE to the folder containing train.py and test.py
REPO = Path(r'')
# Folder created by prepare_pix2pix_loso.py --all
LOSO_ROOT = Path(r'')

# First test just one fold; change RUN_ALL=True to train all 13 folds.
RUN_ALL = False
DATASET = 'HE_TRI'    # 'HE_TRI' or 'HE_PAS'
TEST_SAMPLE = ''

# Original training notebook settings, unchanged by default
DIRECTION = 'AtoB'
GPU_ID = 0

# The standard pix2pix repo saves models into REPO/checkpoints/<name>
MODEL_PREFIX = 'LOSO'

SAMPLES = {
    'HE_TRI': [],
    'HE_PAS': [],
}

def model_name(dataset, sample):
    return f'{MODEL_PREFIX}_{dataset}_test_{sample}'

def fold_path(dataset,sample):
    return LOSO_ROOT / dataset / f'test_{sample}'

print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(GPU_ID) if torch.cuda.is_available() else '(CPU)')
print('Repository:',REPO)
print('LOSO root:',LOSO_ROOT)

## 2. Validate the prepared folds and sample leakage

In [ ]:
assert (REPO / 'train.py').is_file(), f'Missing train.py: {REPO}'
assert (REPO / 'test.py').is_file(), f'Missing test.py: {REPO}'
assert LOSO_ROOT.is_dir(), f'Missing LOSO folder: {LOSO_ROOT}'

JOBS = [(ds,s) for ds in SAMPLES for s in SAMPLES[ds]] 
for ds,heldout in JOBS:
    folder=fold_path(ds,heldout)
    assert folder.is_dir(), f'Missing prepared fold: {folder}'
    rows=list(csv.DictReader((folder/'manifest.csv').open(encoding='utf-8',newline='')))
    by_split={p:{r['sample'] for r in rows if r['split']==p} for p in ('train','val','test')}
    assert by_split['test']=={heldout}, (ds,heldout,by_split)
    assert not (by_split['train'] & by_split['val'])
    assert not (by_split['train'] & by_split['test'])
    assert not (by_split['val'] & by_split['test'])
    for split in ('train','val','test'):
        actual={x.name for x in (folder/split).glob('*.png')}
        expected={r['fold_filename'] for r in rows if r['split']==split}
        assert actual==expected, f'File mismatch: {folder / split}'
    print(f'{ds:6} | held-out {heldout:10} | train={len([r for r in rows if r["split"]=="train"]):4} '
          f'val={len([r for r in rows if r["split"]=="val"]):4} '
          f'test={len([r for r in rows if r["split"]=="test"]):4} '
          f'| train samples={sorted(by_split["train"])}')
print('All chosen folds pass sample-leakage and file checks.')

## 3. Train selected fold(s)

Each fold gets a distinct checkpoint name. The optional `--display_id -1` avoids requiring Visdom. Existing checkpoints with the same names are not overwritten by this notebook.

In [ ]:
from pathlib import Path
import time

# ============================================================
# PATHS
# ============================================================

BASE = Path(
    r""
)

REPO = BASE / "Notebooks" / "pytorch-CycleGAN-and-pix2pix"
TRAIN_SCRIPT = REPO / "train.py"

DATA_ROOT = BASE / "pix2pix_LOSO"

# Correct checkpoint directory
CHECKPOINTS_DIR = BASE / "Notebooks" / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# TRAINING PARAMETERS
# ============================================================

N_EPOCHS = 100
N_EPOCHS_DECAY = 0

BATCH_SIZE = 15
LOAD_SIZE = 286
CROP_SIZE = 256

# Save checkpoints less frequently
SAVE_EPOCH_FREQ = 5

# Reduce logging overhead
PRINT_FREQ = 500

# Skip already completed models
SKIP_COMPLETED = True

# ============================================================
# SAMPLES
# ============================================================

SAMPLES = {
    "HE_TRI": [],

    "HE_PAS": [],
}

# ============================================================
# CHECK PATHS
# ============================================================

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(
        f"Cannot find training script: {TRAIN_SCRIPT}"
    )

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Cannot find LOSO data: {DATA_ROOT}"
    )

# ============================================================
# TRAIN ALL LOSO FOLDS
# ============================================================

TOTAL_FOLDS = sum(len(samples) for samples in SAMPLES.values())

completed = []
skipped = []
failed = []

fold_number = 0

for dataset, samples in SAMPLES.items():

    for sample in samples:

        fold_number += 1

        name = f"LOSO_{dataset}_test_{sample}"

        fold_path = (
            DATA_ROOT
            / dataset
            / f"test_{sample}"
        )

        checkpoint_dir = CHECKPOINTS_DIR / name
        TOTAL_EPOCHS = N_EPOCHS + N_EPOCHS_DECAY

        final_checkpoint = (
            checkpoint_dir / f"{TOTAL_EPOCHS}_net_G.pth"
        )

        print("\n" + "=" * 70)
        print(f"FOLD {fold_number}/{TOTAL_FOLDS}")
        print(f"Dataset: {dataset}")
        print(f"Held-out sample: {sample}")
        print(f"Experiment: {name}")
        print("=" * 70)

        # ----------------------------------------------------
        # SKIP COMPLETED TRAINING
        # ----------------------------------------------------

        if SKIP_COMPLETED and final_checkpoint.exists():

            print("Already completed. Skipping.")

            skipped.append(name)
            continue

        # ----------------------------------------------------
        # VERIFY TRAINING DATA
        # ----------------------------------------------------

        train_path = fold_path / "train"
        val_path = fold_path / "val"
        test_path = fold_path / "test"

        for path in [train_path, val_path, test_path]:

            if not path.exists():
                raise FileNotFoundError(
                    f"Missing folder: {path}"
                )

            n_images = len(list(path.glob("*.png")))

            print(f"{path.name}: {n_images} images")

            if n_images == 0:
                raise RuntimeError(
                    f"No images found in {path}"
                )

        # ----------------------------------------------------
        # START TRAINING
        # ----------------------------------------------------

        print("\nStarting training...\n", flush=True)

        start_time = time.time()

        # SAME JUPYTER COMMAND FORMAT AS YOUR WORKING TEST

        !python "{TRAIN_SCRIPT}" \
            --dataroot "{fold_path}" \
            --checkpoints_dir "{CHECKPOINTS_DIR}" \
            --name "{name}" \
            --model pix2pix \
            --direction AtoB \
            --dataset_mode aligned \
            --n_epochs {N_EPOCHS} \
            --n_epochs_decay {N_EPOCHS_DECAY} \
            --batch_size {BATCH_SIZE} \
            --load_size {LOAD_SIZE} \
            --crop_size {CROP_SIZE} \
            --save_latest_freq 5000 \
            --save_epoch_freq {SAVE_EPOCH_FREQ} \
            --print_freq {PRINT_FREQ} \
            --verbose
        # ----------------------------------------------------
        # VERIFY COMPLETION
        # ----------------------------------------------------

        elapsed = (time.time() - start_time) / 60

        if final_checkpoint.exists():

            print(f"\nSuccessfully completed: {name}")
            print(f"Training time: {elapsed:.1f} minutes")
            print(f"Checkpoint: {final_checkpoint}")

            completed.append(name)

        else:

            print(f"\nTraining failed or was interrupted: {name}")

            failed.append(name)

            raise RuntimeError(
                f"Final checkpoint not found:\n"
                f"{final_checkpoint}\n"
                f"Check the pix2pix output above."
            )

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("LOSO TRAINING SUMMARY")
print("=" * 70)

print(f"Newly completed: {len(completed)}")
print(f"Previously completed: {len(skipped)}")
print(f"Failed: {len(failed)}")

print("\nCompleted models:")
for name in completed:
    print("  ", name)

print("\nSkipped models:")
for name in skipped:
    print("  ", name)

if not failed:
    print("\nALL LOSO TRAINING FOLDS COMPLETED!")

## 4. Predict only on held-out test samples

**Run after training.** The source image pairs were created at 512×512, while the original notebook trained with 286→256 transforms. For comparable 256×256 inference without randomness, this uses `--preprocess resize --load_size 256 --crop_size 256`. The predictions will therefore be **256×256**, not native 512×512 mosaics. You can later change the training regime to native 512×512 if desired.

In [ ]:
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

BASE = Path(
    r""
)

REPO = BASE / "Notebooks" / "pytorch-CycleGAN-and-pix2pix"
DATA_ROOT = BASE / "pix2pix_LOSO"

CHECKPOINTS_DIR = BASE / "Notebooks" / "checkpoints"
RESULTS_DIR = BASE / "Notebooks" / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================

RUN_INFERENCE = True

NUM_TEST = 1000000

# Use the final saved epoch.
# You trained with 100 epochs, so use 100.
EPOCH = "100"

# ============================================================
# SAMPLES
# ============================================================

SAMPLES = {
    "HE_TRI": [
    ],

    "HE_PAS": [
    ],
}

# ============================================================
# RUN INFERENCE
# ============================================================

if RUN_INFERENCE:

    for dataset, samples in SAMPLES.items():

        for sample in samples:

            name = f"LOSO_{dataset}_test_{sample}"

            fold_path = DATA_ROOT / dataset / f"test_{sample}"

            checkpoint = (
                CHECKPOINTS_DIR
                / name
                / f"{EPOCH}_net_G.pth"
            )

            print("\n" + "=" * 70)
            print(f"INFERENCE: {dataset} | Test sample: {sample}")
            print("=" * 70)

            if not checkpoint.exists():
                print(f"Checkpoint not found. Skipping: {checkpoint}")
                continue

            if not (fold_path / "test").exists():
                print(f"Test folder missing. Skipping: {fold_path}")
                continue

            n_test = len(list((fold_path / "test").glob("*.png")))

            print(f"Test images: {n_test}")
            print(f"Checkpoint: {checkpoint}")

            # Same execution method as working training code
            !python "{REPO}/test.py" \
                --dataroot "{fold_path}" \
                --checkpoints_dir "{CHECKPOINTS_DIR}" \
                --results_dir "{RESULTS_DIR}" \
                --name "{name}" \
                --model pix2pix \
                --dataset_mode aligned \
                --direction AtoB \
                --phase test \
                --epoch "{EPOCH}" \
                --num_test {NUM_TEST} \
                --preprocess resize \
                --load_size 256 \
                --crop_size 256 \
                --no_dropout \

            images_dir = (
                RESULTS_DIR
                / name
                / f"test_{EPOCH}"
                / "images"
            )

            print(f"\nResults: {images_dir}")

else:
    print("Inference disabled.")

## 5. Inspect generated images from a held-out sample

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

VIEW_DATASET = DATASET
VIEW_SAMPLE = TEST_SAMPLE
N_SHOW = 4
p=REPO/'results'/model_name(VIEW_DATASET,VIEW_SAMPLE)/'test_latest'/'images'
if not p.is_dir():
    print('No inference images yet:',p)
else:
    fake_paths=sorted(p.glob('*_fake_B.png'))[:N_SHOW]
    if not fake_paths:
        print('No fake_B images in',p)
    else:
        fig,axes=plt.subplots(len(fake_paths),3,figsize=(11,3.5*len(fake_paths)),squeeze=False)
        for i,fake_path in enumerate(fake_paths):
            prefix=fake_path.name[:-len('_fake_B.png')]
            for j,(suffix,label) in enumerate([('_real_A.png','H&E'),('_fake_B.png','Generated stain'),('_real_B.png','Ground truth')]):
                with Image.open(p/(prefix+suffix)) as im:
                    axes[i,j].imshow(im.convert('RGB'))
                axes[i,j].set_title(label)
                axes[i,j].axis('off')
        plt.tight_layout()
        plt.show()